# SOXX 历史期权链：已有 Alpha Vantage 凭据权限预检

仅一次固定2011日期的只读 GET；若权限不足，停止第二日期。不得购买、拟合或运行 Stage1 账户。

In [1]:
import os,sys,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import _settings,nas_api_call
card=root/'research/stage1_alpha_vantage_soxx_option_history_preflight_20261023';snap=root/'data/snapshots/stage1_alpha_vantage_soxx_option_history_preflight_20261023';plan=json.loads((card/'plan.json').read_text());receipt=json.loads((card/'request_receipt_1.json').read_text());state=json.loads((card/'run_state.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and state['HTTP_GET_attempts']==1 and state['plan_sha256']==sha(card/'plan.json') and receipt['request_attempts']==1 and receipt['date']==plan['dates_in_order'][0]
raw=snap/'2011-10-03_body.bin';assert raw.is_file() and sha(raw)==receipt['body_sha256'] and raw.stat().st_size==receipt['body_bytes'] and receipt['HTTP_status']==200
content=raw.read_bytes();assert os.environ['ALPHA_VANTAGE_API_KEY'].encode() not in content
response=json.loads(content);assert list(response)==['Information'] and isinstance(response['Information'],str) and 'premium' in response['Information'].lower() and 'subscribe' in response['Information'].lower()
result={'decision':'existing_Alpha_Vantage_key_HISTORICAL_OPTIONS_premium_entitlement_denied','plan_sha256':sha(card/'plan.json'),'HTTP_GET_attempts':1,'requested_date':plan['dates_in_order'][0],'second_date_requested':False,'HTTP_status':200,'body_sha256':sha(raw),'response_keys':list(response),'premium_entitlement_message_observed':True,'option_chain_rows':0,'SOXX_specific_2011_history_qualified':False,'IV_Delta_fields_qualified':False,'first_public_clock_qualified':False,'new_purchases':0,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
settings=_settings(root);members=[card/'plan.json',card/'request_receipt_1.json',card/'result.json',raw]
with tempfile.TemporaryDirectory(prefix='soxx-av-options-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for p in members:out.add(p,arcname=str(p.relative_to(root)))
 digest=sha(archive);response_api=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research');uploaded=response_api['artifact'];assert uploaded['sha256']==digest and uploaded['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;restored=Path(tmp)/'restored.tar.gz'
 with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(restored)==digest
 with tarfile.open(restored,'r:gz') as z:
  names={m.name:m for m in z.getmembers() if m.isfile()};assert set(names)=={str(p.relative_to(root)) for p in members}
  for p in members:assert hashlib.sha256(z.extractfile(names[str(p.relative_to(root))]).read()).hexdigest()==sha(p)
archive_receipt={'sha256':digest,'size_bytes':uploaded['size_bytes'],'nas_path':remote,'members':{str(p.relative_to(root)):sha(p) for p in members},'remote_and_scoped_restore_verified':True};(card/'nas_source_archive.json').write_text(json.dumps(archive_receipt,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_archive':'nas_source_archive.json','HTTP_GET_attempts':1,'new_purchases':0,'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Existing credential does not entitle HISTORICAL_OPTIONS; a 200 premium information body is not a SOXX option chain.','No vendor purchase or second historical date request was made.','No SOXX skew factor, information test or Stage1 account.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['phase']='finished_entitlement_denied';state['NAS_sha256']=digest;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print({k:result[k] for k in ('decision','HTTP_GET_attempts','second_date_requested','option_chain_rows','new_purchases','model_fits','native_accounts')})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'decision': 'existing_Alpha_Vantage_key_HISTORICAL_OPTIONS_premium_entitlement_denied', 'HTTP_GET_attempts': 1, 'second_date_requested': False, 'option_chain_rows': 0, 'new_purchases': 0, 'model_fits': 0, 'native_accounts': 0}


# SOXX 历史期权链：已有 Alpha Vantage 凭据权限预检

[Alpha Vantage 官方 API 文档](https://www.alphavantage.co/documentation/#historical-options)提供 `HISTORICAL_OPTIONS`，描述为按日期返回期权链及 IV/Greeks，并明确属于 premium。按冻结计划，只以当前环境已有凭据读取 SOXX 2011-10-03 一次，不购买、不接触供应商。HTTP 为 200，但原始 JSON 只有 `Information`，内容要求订阅 premium；**零合约、零 IV/Delta**。因此没有请求预定的第二个 2024 日期。HTTP 200 不是数据授权通过。

响应原字节 SHA-256 `6c7d1922…`、请求收据和冻结计划均已归档 Yanxuan NAS，归档 SHA-256 `8aaa7727…` 完成远端哈希与限定恢复；回执和 Git 工件都不含 API key。零 SOXX 未来收益读取、零模型拟合、零账户、零购买。

**决定：关闭当前 Alpha Vantage 凭据的历史期权权限路径。** 这不证明 SOXX 2011–2026 期权历史不存在，也不证明购买 premium 后一定有 SOXX 全期覆盖、双边报价时刻或可用的25Δ恒定期限样本。历史 SOXX/SMH 期权链仍需先取得有权限的具体日期样本及首发/修订/缺失审计；现有 ORATS 只询样草稿尚待用户授权发送，不能据此运行偏斜信息门或 Stage1 0bp账户。
